# Binary Intrusion Detection

This notebook trains models to classify network connections as `normal` or `attack`. The official NSL-KDD test split is used for evaluation to measure generalization to unseen attack patterns.


In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)


/Users/paco/Documents/Git Projects/network-intrusion-detection-system


In [2]:
import json

import pandas as pd

from src.config import CLEANED_DIR, FEATURE_COLUMNS, MODELS_DIR, REPORTS_DIR, FIGURES_DIR, ensure_project_dirs
from src.modeling import binary_models, evaluate_binary, metrics_to_frame, save_model
from src.visualize import save_metric_barplot

ensure_project_dirs()
train_df = pd.read_csv(CLEANED_DIR / "train_cleaned.csv")
test_df = pd.read_csv(CLEANED_DIR / "test_cleaned.csv")

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["binary_label"]
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df["binary_label"]


## Train and Evaluate Models


In [3]:
models = binary_models()
results = {}
fitted_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    fitted_models[name] = model
    results[name] = evaluate_binary(model, X_test, y_test)

binary_results = metrics_to_frame(results).sort_values("f1", ascending=False)
display(binary_results)
binary_results.to_csv(REPORTS_DIR / "binary_model_comparison.csv", index=False)
save_metric_barplot(binary_results, "f1", FIGURES_DIR / "binary_f1_comparison.png", "Binary Intrusion Detection - F1 Score")


Training Logistic Regression...


/Users/paco/Documents/Git Projects/network-intrusion-detection-system/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training Random Forest...
Training HistGradientBoosting...


,model,accuracy,precision,recall,f1,roc_auc
2,HistGradientBoosting,0.797285,0.968690,0.665394,0.788895,0.960684
1,Random Forest,0.777502,0.968364,0.629705,0.763150,0.962321
0,Logistic Regression,0.754746,0.916990,0.625808,0.743921,0.790872


## Save Best Binary Model


In [4]:
best_binary_name = binary_results.iloc[0]["model"]
best_binary_model = fitted_models[best_binary_name]
save_model(best_binary_model, MODELS_DIR / "binary_intrusion_model.pkl")

with open(REPORTS_DIR / "binary_metrics.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=2)

print("Best binary model:", best_binary_name)


Best binary model: HistGradientBoosting


## Business Interpretation


In [5]:
binary_results[["model", "precision", "recall", "f1", "roc_auc"]]


,model,precision,recall,f1,roc_auc
2,HistGradientBoosting,0.968690,0.665394,0.788895,0.960684
1,Random Forest,0.968364,0.629705,0.763150,0.962321
0,Logistic Regression,0.916990,0.625808,0.743921,0.790872
